# Lab 4.1 &mdash; The Tool Contract

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 2 &middot; Module 4 &mdash; Tool Calling &amp; MCP**

### What you'll do
- See the exact JSON a <code>@tool</code> becomes, and what it leaves behind
- Write the argument descriptions the model reads to fill a call in
- Decide which failures are worth retrying &mdash; and which never are
- Wire <code>handle_tool_error</code> so a failing tool returns instead of raising

> **How this lab works.** You write real LangChain and MCP code. Fill every `BLANK`, then run
> the **Self-check** cell under each section &mdash; those assert on the *objects you built*
> (a `@tool`, an argument schema, a `ToolMessage`, an `mcp.types.Tool`), so they are
> deterministic and do not depend on the model. Cells marked **Run it for real** put your code
> in front of the sandbox model; that is the part worth watching. The score line is feedback,
> not a grade.

> **Start here.** Everything else in Module 4 &mdash; selection accuracy, multi-tool
> orchestration, MCP &mdash; is this contract, either written by you or by someone else.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-4-01")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model can reason before it answers, and that reasoning is billed as completion
# tokens. It is off here because tool selection is a short decision and you will make a lot
# of them today. Pass think=True to see the difference for yourself.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 4 labs -- the same payment exceptions as Day 1,
# now reached through tools the model chooses, and then through tools you did not write.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

In [ ]:
# ------------------------------------------------- the toolkit (nothing to fill in)
# Four tools over that ledger, written with LangChain's @tool decorator. Three read; one
# moves money -- the distinction that starts mattering the moment a model is choosing.
# Read the docstrings properly: they are not comments, they are the API the model sees.
from langchain_core.tools import tool

@tool
def lookup_payment(ref: str) -> str:
    """Return the ledger record for one payment reference such as 'PMT-1002'.

    Use when you already have the reference. Not for searching across payments --
    use search_payments when you do not have one.
    """
    record = LEDGER.get(ref)
    if record is None:
        return f"no payment found with reference {ref!r}"
    return json.dumps({"ref": ref, **record})


@tool
def search_payments(counterparty: str = "", status: str = "") -> str:
    """Return every ledger record matching a counterparty, a status, or both.

    Use when you must find which payments match. Not for one known reference --
    use lookup_payment for that.
    """
    hits = [{"ref": r, **v} for r, v in LEDGER.items()
            if (not counterparty or v["counterparty"] == counterparty)
            and (not status or v["status"] == status)]
    return json.dumps(hits)


@tool
def policy_for(reason_code: str) -> str:
    """Return the operating policy for one failure reason code such as 'LIMIT_BREACH'.

    Use once you know why a payment failed and need to know what to do about it.
    """
    return POLICY.get(reason_code, f"no policy on file for reason code {reason_code!r}")


@tool
def release_payment(ref: str) -> str:
    """Release one held payment so that it settles. This one moves money.

    Use only after a named human has approved this specific release. Not for reading,
    searching or explaining.
    """
    record = LEDGER.get(ref)
    if record is None:
        return f"no payment found with reference {ref!r}"
    return json.dumps({"ref": ref, "released": True, "was": record["status"]})


TOOLKIT = [lookup_payment, search_payments, policy_for, release_payment]
BY_NAME = {t.name: t for t in TOOLKIT}
print("toolkit:", ", ".join(BY_NAME))

## Concept

A tool is not the function you wrote. From the model's side a tool is exactly three fields:

| field | comes from | what it decides |
|---|---|---|
| `name` | the function name | how the tool is referred to |
| `description` | the docstring | **whether it is chosen at all** |
| `parameters` | the signature and `args_schema` | whether the arguments are well formed |

The body, the tests and the types you were careful about never cross the boundary. That is the
whole reason a tool-calling bug is usually a writing bug.

## Section 1 &mdash; What actually crosses the boundary

`convert_to_openai_tool` renders a LangChain tool into the exact JSON that goes on the wire.
Look at it once and you stop guessing about the rest of the module.

In [ ]:
from langchain_core.utils.function_calling import convert_to_openai_tool

def what_the_model_sees(t) -> dict:
    """The exact JSON one tool becomes on the wire. Nothing else about it is sent."""
    return convert_to_openai_tool(t)["function"]


def selection_text(t) -> str:
    """The one field a model reads when deciding whether to call this tool at all.

    The name says how the tool is REFERRED to. The parameters say what a well-formed call
    looks like. Neither of them says when to call it.
    """
    return BLANK          # TODO: which field of the tool decides whether it gets chosen?

In [ ]:
# --- Self-check: Section 1   (tool objects only -- no model call)
check("a @tool is a framework object, not a plain function",
      lambda: hasattr(lookup_payment, "name") and hasattr(lookup_payment, "args_schema"))
check("the wire form carries the three fields that cross the boundary",
      lambda: {"name", "description", "parameters"} <= set(what_the_model_sees(lookup_payment)))
check("the name is the function's own name",
      lambda: what_the_model_sees(lookup_payment)["name"] == "lookup_payment")
check("the description is the WHOLE docstring, not just its first line",
      lambda: "Not for searching" in selection_text(lookup_payment),
      "the boundary sentence is the part that stops the wrong call -- it must not be truncated")
check("and it is the same text that goes on the wire",
      lambda: what_the_model_sees(lookup_payment)["description"] == selection_text(lookup_payment))
check("an argument with no default is required",
      lambda: what_the_model_sees(lookup_payment)["parameters"]["required"] == ["ref"])
check("an argument with a default is optional",
      lambda: "counterparty" not in
              (what_the_model_sees(search_payments)["parameters"].get("required") or []))
check("the implementation does not cross the boundary",
      lambda: "LEDGER" not in json.dumps(what_the_model_sees(lookup_payment)),
      "the model never sees the body -- your careful code is invisible to the choice")

guard(lambda: print(json.dumps(what_the_model_sees(lookup_payment), indent=2)[:520]))

## Section 2 &mdash; The schema is prose too

The parameters are not just types. Every field can carry a `description`, and those descriptions
go to the model with the rest of the schema. An optional argument whose default is never stated
is one the model guesses at.

Attach a hand-written `args_schema` to `search_payments` and write those descriptions yourself.

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.tools import StructuredTool

class SearchArgs(BaseModel):
    """Find payments when you do not have a reference."""

    # TODO: replace "BLANK" with what a model needs in order to fill this in.
    #       Name the kind of value it takes, and say what leaving it empty means.
    counterparty: str = Field(default="", description="BLANK")

    status: str = Field(
        default="",
        description="One of: settled, failed, held. Empty means any status.")


def described_search() -> StructuredTool:
    """The same function, now with your argument schema attached to it."""
    return StructuredTool.from_function(
        func=search_payments.func,
        name="search_payments",
        description=search_payments.description,
        args_schema=SearchArgs)

In [ ]:
# --- Self-check: Section 2   (schema objects only -- no model call)
def _desc(field: str) -> str:
    """The description on one field. An untouched placeholder is a TODO, not a failure."""
    d = (SearchArgs.model_fields[field].description or "").strip()
    if d == "BLANK":
        raise NameError(f"{field} still has the placeholder description")
    return d

check("every argument carries a description the model can read",
      lambda: all(_desc(f) for f in SearchArgs.model_fields))
check("the counterparty description says what an empty value means",
      lambda: "empty" in _desc("counterparty").lower(),
      "an optional argument whose default is unstated is one the model guesses at")
check("it shows how the ledger actually spells a counterparty",
      lambda: any(n in _desc("counterparty") for n in {v["counterparty"] for v in LEDGER.values()}),
      "name one of NORTHWIND / ACME-EU / ZENITH / ACME-UK -- the model cannot guess your spelling")
check("the status description names the values it accepts",
      lambda: all(s in _desc("status") for s in ("settled", "failed", "held")))
check("the schema really is attached to the tool",
      lambda: set(described_search().args) == set(SearchArgs.model_fields))
check("both arguments reach the wire",
      lambda: set(what_the_model_sees(described_search())["parameters"]["properties"])
              == {"counterparty", "status"})
check("and so do your descriptions of them",
      lambda: what_the_model_sees(described_search())["parameters"]["properties"]
              ["counterparty"]["description"] == _desc("counterparty"))

## Section 3 &mdash; A tool that returns instead of raising

Module 2 called blind retry the most common production failure. The fix starts here: a result
that says *whether trying again could possibly help*.

Retrying a malformed argument produces the same malformed argument. Retrying a permission denial
produces the same denial. Only a **transient** failure has earned a second attempt.

`ToolException` plus `handle_tool_error` is how LangChain turns a raise into a string the model
reads as an observation.

In [ ]:
from langchain_core.tools import ToolException

ERROR_KINDS = ("not_found", "invalid_input", "unavailable", "timeout", "not_permitted")

def is_retryable(kind: str) -> bool:
    """True only for failures where the identical call might succeed on a second attempt."""
    # TODO: which of ERROR_KINDS are transient? Retrying a bad argument sends the same bad
    #       argument; retrying a denial is how you page a security team at 3am.
    return BLANK


def _strict_lookup(ref: str, ledger_down: bool = False) -> str:
    """Return the ledger record for one payment reference such as 'PMT-1002'.

    Use when you already have the reference. Not for searching across payments.
    """
    if not ref.startswith("PMT-"):
        raise ToolException(f"invalid_input: {ref!r} is not a payment reference")
    if ledger_down:
        raise ToolException("unavailable: the ledger service did not respond")
    if ref not in LEDGER:
        # "I looked and it is not there" is not the same fact as "I could not look".
        raise ToolException(f"not_found: no payment on file with reference {ref!r}")
    return json.dumps({"ref": ref, **LEDGER[ref]})


def describe_failure(exc: ToolException) -> str:
    """What the model is told when the tool fails. It reads this as an observation."""
    kind = str(exc).split(":")[0]
    return json.dumps({"error": kind, "message": str(exc), "retryable": is_retryable(kind)})


def safe_lookup() -> StructuredTool:
    """The same lookup, wrapped so a failure comes back as text the agent can act on."""
    return StructuredTool.from_function(
        func=_strict_lookup,
        name="lookup_payment",
        description=lookup_payment.description,
        handle_tool_error=describe_failure)

In [ ]:
# --- Self-check: Section 3   (running a tool is plain Python -- still no model call)
def _out(**kwargs) -> dict:
    """Invoke the wrapped tool and read whatever came back as JSON."""
    return json.loads(safe_lookup().invoke(kwargs))

check("a service that did not answer is worth another try",
      lambda: is_retryable("unavailable") is True)
check("so is a timeout", lambda: is_retryable("timeout") is True)
check("a bad argument is not -- the retry sends the same bad argument",
      lambda: is_retryable("invalid_input") is False)
check("a missing record is not -- it will still be missing",
      lambda: is_retryable("not_found") is False)
check("a permission denial is not -- and retrying it is how you page a security team",
      lambda: is_retryable("not_permitted") is False)

check("a known payment comes back as the record",
      lambda: _out(ref="PMT-1002")["reason_code"] == "INSUFFICIENT_FUNDS")
check("a malformed reference comes back as TEXT, not as an exception",
      lambda: _out(ref="northwind")["error"] == "invalid_input",
      "handle_tool_error is what turns the raise into something the agent can read")
check("a well-formed reference that is absent is not_found",
      lambda: _out(ref="PMT-9999")["error"] == "not_found")
check("a down ledger is unavailable -- and is the only one of the three worth retrying",
      lambda: _out(ref="PMT-1002", ledger_down=True)["error"] == "unavailable"
              and _out(ref="PMT-1002", ledger_down=True)["retryable"] is True)
check("'not there' and 'could not look' stay different answers",
      lambda: _out(ref="PMT-9999")["error"] != _out(ref="PMT-9999", ledger_down=True)["error"],
      "collapse these and the agent reports a payment missing when the ledger merely blinked")

for probe in ({"ref": "PMT-1002"}, {"ref": "PMT-9999"}, {"ref": "northwind"},
              {"ref": "PMT-1002", "ledger_down": True}):
    guard(lambda p=probe: print(f"  {str(p):42} -> {safe_lookup().invoke(p)[:74]}"))

## Run it for real

Two things at once. First, bind the tool to the model and watch a `tool_call` come back &mdash;
that is the contract being used, not described. Then hand the model the descriptor and nothing
else, and ask what the tool is for and when it should *not* be used.

In [ ]:
if llm_ready():
    def _probe():
        bound = get_llm().bind_tools([lookup_payment])
        reply = bound.invoke("What is the status of PMT-1002?")
        print("  tool_calls:", reply.tool_calls)
        print()
        print(ask("Here is a tool available to an agent, in the exact form the agent receives "
                  "it.\n\n" + json.dumps(what_the_model_sees(lookup_payment), indent=2)
                  + "\n\nIn two sentences: what is this tool for, and when should it NOT be "
                    "used?").strip()[:460])
    guard(_probe)

### Read it

The `tool_call` came back as a dict with a `name`, an `args` and an `id`. You did not parse any
text to get it &mdash; the model emitted a structured call because the schema told it what one
looks like. Lab 4.3 turns that into a loop.

The second half is the description doing its job in slow motion. The model has no more
information than you gave it. Anything it gets wrong here, it would also get wrong while choosing
between four tools under time pressure &mdash; except that there you would never see it reason
about it.

In [ ]:
score()

## Your turn

1. Delete the second paragraph of `lookup_payment`'s docstring, re-run Section 1, and read the
   wire form again. Nothing errors. What exactly did you just remove from the model's view?
2. Give `SearchArgs.status` a `Literal["settled", "failed", "held"]` type instead of `str` and
   look at the schema again. Which is the stronger control &mdash; the prose or the type &mdash;
   and which one still lets the model pass `"HELD"`?
3. Add a `not_permitted` path to `_strict_lookup` for a reference outside an allowed range.
   Which of the five failure kinds should an agent be allowed to repeat to the user verbatim,
   and which should it summarise?